# Rally E2B Scorecard

Score direct Gemma4 E2B Heretic against the A100/B75 RP candidate on Kaggle, without browser WebGPU load on a local desktop.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
except Exception as exc:
    print('torch_probe_error=', repr(exc))


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=0.34.0',
    'peft>=0.12.0',
    'safetensors>=0.4.5',
    'huggingface_hub[hf_transfer]>=0.24.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('RALLY_SCORECARD_WORK_DIR', '/kaggle/working/rally-e2b-scorecard'))
REPORT_PATH = WORK_DIR / 'rally-e2b-scorecard-report.json'
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_e2b_scorecard.py'),
    '--work-dir', str(WORK_DIR),
    '--report-path', str(REPORT_PATH),
    '--artifact-name', os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e2b-two-stage-sft'),
    '--direct-model-id', os.environ.get('RALLY_DIRECT_MODEL_ID', 'p-e-w/gemma-4-E2B-it-heretic-ara'),
    '--candidate-name', os.environ.get('RALLY_CANDIDATE_NAME', 'a100-b75'),
    '--stage-b-scale', os.environ.get('RALLY_STAGE_B_SCALE', '0.75'),
    '--max-new-tokens', os.environ.get('RALLY_SCORECARD_MAX_TOKENS', '32'),
    '--temperature', os.environ.get('RALLY_SCORECARD_TEMPERATURE', '0.2'),
    '--min-total', os.environ.get('RALLY_SCORECARD_MIN_TOTAL', '0.70'),
    '--min-margin', os.environ.get('RALLY_SCORECARD_MIN_MARGIN', '0.05'),
]
if os.environ.get('RALLY_KEEP_MERGED', '0') == '1':
    cmd.append('--keep-merged')
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json, os

report_path = Path(os.environ.get('RALLY_SCORECARD_WORK_DIR', '/kaggle/working/rally-e2b-scorecard')) / 'rally-e2b-scorecard-report.json'
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'artifact_dir': report.get('artifact_dir'),
        'direct_total': report.get('scores', {}).get('direct', {}).get('total'),
        'rp_total': report.get('scores', {}).get('rp', {}).get('total'),
        'promotion_decision': report.get('promotion_decision'),
    }, indent=2))
else:
    raise FileNotFoundError(report_path)
